# Skin Lesion Classification on HAM10000

Comparative study of **ResNet-50**, **EfficientNet-B3** and **ViT-B/16** for
seven-class skin lesion classification, with a soft-voting ensemble, statistical
testing (McNemar), calibration analysis and Grad-CAM interpretability.

All pipeline logic lives in the `src/` package; this notebook is the runnable
narrative that calls it.

## 1. Environment setup

In [ ]:
# Make the `src` package importable.
#   - Locally: launch Jupyter from the repo root, nothing to do here.
#   - On Colab: clone the repo (set REPO_URL once it is on GitHub). If you have
#     instead uploaded/unzipped the project so that a `src/` folder already sits
#     in the working directory, the clone step is skipped automatically.
import os
import sys

REPO_URL = "https://github.com/<your-username>/skin-lesion-classification.git"

if "google.colab" in sys.modules:
    if not os.path.isdir("src"):
        repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
        if not os.path.isdir(repo):
            !git clone $REPO_URL
        %cd $repo
    # Colab already ships torch / numpy / pandas / sklearn -- install only the extras:
    !pip install -q timm "albumentations==1.3.1" grad-cam statsmodels pyyaml

## 2. Imports and configuration

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import src
from src import (Config, set_seed, get_device, describe_device, setup_logging,
                 load_metadata, group_split, build_model, count_trainable_params,
                 train_model, lr_sweep, compute_metrics, mcnemar_test,
                 significance_stars, expected_calibration_error, soft_vote)
from src import viz
from src.interpretability import gradcam_for_model

setup_logging()
cfg = Config.from_yaml("configs/default.yaml")
cfg.make_dirs()
set_seed(cfg.seed)
device = get_device()
print(f"src v{src.__version__}  |  device: {describe_device(device)}")

## 3. Data

The dataset is fetched with `scripts/download_data.py`, which reads your Kaggle
token from `~/.kaggle/kaggle.json`.

In [ ]:
# Kaggle credentials are needed once to download the dataset (Colab).
if "google.colab" in sys.modules and not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    from google.colab import files
    print("Upload kaggle.json:")
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!python scripts/download_data.py --data-dir {cfg.data_dir}

## 4. Exploratory data analysis

In [ ]:
df = load_metadata(cfg)
df.head()

In [ ]:
counts = df["dx"].value_counts().reindex(cfg.class_names)
print("Class distribution:")
print(counts)
print(f"\nImbalance ratio (max/min): {counts.max() / counts.min():.1f}")

viz.plot_class_distribution(df, cfg);

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print(f"\nAge: min={df['age'].min()}, max={df['age'].max()}, "
      f"mean={df['age'].mean():.1f}, missing={int(df['age'].isna().sum())}")
print(f"Histopathology-confirmed: {(df['dx_type'] == 'histo').mean() * 100:.1f}%")

In [ ]:
viz.plot_demographics(df, cfg)
viz.plot_images_per_lesion(df, cfg);

In [ ]:
viz.plot_sample_per_class(df, cfg);

## 5. Data preparation (leakage-free split)

The split is grouped by `lesion_id` so that multiple images of the same lesion
never span train/val/test.

In [ ]:
df_train, df_val, df_test = group_split(df, cfg)
print(f"Train: {len(df_train)} images, {df_train['lesion_id'].nunique()} lesions")
print(f"Val:   {len(df_val)} images, {df_val['lesion_id'].nunique()} lesions")
print(f"Test:  {len(df_test)} images, {df_test['lesion_id'].nunique()} lesions")

split_dist = pd.DataFrame({
    "Train": df_train["dx"].value_counts(normalize=True),
    "Val":   df_val["dx"].value_counts(normalize=True),
    "Test":  df_test["dx"].value_counts(normalize=True),
}).reindex(cfg.class_names) * 100
print("\nClass distribution by split (%):")
print(split_dist.round(2))

## 6. Models

Each backbone is loaded pretrained on ImageNet with the body frozen; only the
classifier head is trained, keeping the comparison fair under a fixed budget.

In [ ]:
print(f"{'Model':<25} {'Trainable':>12} {'Total':>14}")
for name in cfg.model_names:
    model = build_model(name, cfg.num_classes, device)
    trainable, total = count_trainable_params(model)
    print(f"{cfg.model_display[name]:<25} {trainable:>12,} {total:>14,}")
    del model
    torch.cuda.empty_cache()

## 7. Training

Per-architecture learning rates, AMP with gradient clipping, cosine schedule and
early stopping on validation balanced accuracy.

In [ ]:
results = {}
for name in cfg.model_names:
    results[name] = train_model(cfg, name, df_train, df_val, df_test, device,
                                lr=cfg.lr_for(name))
    torch.cuda.empty_cache()

with open(cfg.output_dir / "results_main.json", "w") as f:
    json.dump({n: r["metrics"] for n, r in results.items()}, f, indent=2)

## 8. Learning-rate sweep (EfficientNet-B3)

In [ ]:
sweep_df = lr_sweep(cfg, "efficientnet_b3", df_train, df_val, df_test, device)
print(sweep_df.round(4))
sweep_df.to_csv(cfg.output_dir / "lr_sweep.csv", index=False)
viz.plot_lr_sweep(sweep_df, cfg, "efficientnet_b3");

## 9. Results and comparison

In [ ]:
comp_df = viz.build_comparison_table(results, cfg)
print(comp_df.round(4).to_string(index=False))
comp_df.to_csv(cfg.output_dir / "comparison_table.csv", index=False)
viz.plot_model_comparison(comp_df, cfg);

In [ ]:
viz.plot_confusion_matrices(results, cfg);

## 10. Statistical significance (McNemar)

In [ ]:
print(f"{'Model A':<20} {'Model B':<20} {'A only':>8} {'B only':>8} {'p-value':>10} {'Sig.':>5}")
print("-" * 80)
labels_arr = np.array(results[cfg.model_names[0]]["test_labels"])
mcnemar_rows = []
for i in range(len(cfg.model_names)):
    for j in range(i + 1, len(cfg.model_names)):
        a, b = cfg.model_names[i], cfg.model_names[j]
        r = mcnemar_test(np.array(results[a]["test_preds"]),
                         np.array(results[b]["test_preds"]), labels_arr)
        sig = significance_stars(r["pvalue"])
        mcnemar_rows.append({"model_a": cfg.model_display[a],
                             "model_b": cfg.model_display[b], **r, "significance": sig})
        print(f"{cfg.model_display[a]:<20} {cfg.model_display[b]:<20} "
              f"{r['b1_a_only']:>8} {r['b2_b_only']:>8} {r['pvalue']:>10.4f} {sig:>5}")
pd.DataFrame(mcnemar_rows).to_csv(cfg.output_dir / "mcnemar.csv", index=False)

## 11. Per-class performance

In [ ]:
pc_df = viz.per_class_table(results, cfg)
print(pc_df.round(4).to_string(index=False))
pc_df.to_csv(cfg.output_dir / "per_class.csv", index=False)
viz.plot_per_class_f1(pc_df, cfg);

## 12. Error analysis

The most confidently wrong predictions of the best model.

In [ ]:
viz.plot_misclassified(results, df_test, cfg);

## 13. Calibration

In [ ]:
for name, res in results.items():
    ece = expected_calibration_error(np.array(res["test_labels"]), np.array(res["test_probs"]))
    print(f"{cfg.model_display[name]:<20} ECE = {ece:.3f}")
viz.plot_calibration(results, cfg);

## 14. Ensemble (soft voting)

In [ ]:
ensemble_probs, ensemble_preds = soft_vote(results, cfg.model_names)
labels_arr = np.array(results[cfg.model_names[0]]["test_labels"])
ensemble_metrics = compute_metrics(labels_arr, ensemble_preds, ensemble_probs, cfg.class_names)

print("Ensemble (soft voting):")
for k in ("accuracy", "balanced_accuracy", "f1_weighted", "auc_macro"):
    print(f"  {k:<18} {ensemble_metrics[k]:.4f}")

ens_row = pd.DataFrame([{
    "Model": "Ensemble (soft vote)",
    "Accuracy": ensemble_metrics["accuracy"],
    "Balanced Accuracy": ensemble_metrics["balanced_accuracy"],
    "F1 (weighted)": ensemble_metrics["f1_weighted"],
    "F1 (macro)": ensemble_metrics["f1_macro"],
    "AUC (macro)": ensemble_metrics["auc_macro"],
    "Train time (s)": None, "Throughput (img/s)": None,
}])
final_df = pd.concat([comp_df, ens_row], ignore_index=True)
final_df.to_csv(cfg.output_dir / "comparison_with_ensemble.csv", index=False)
print()
print(final_df.round(4).to_string(index=False))

## 15. Interpretability (Grad-CAM)

In [ ]:
for name in ["resnet50", "efficientnet_b3"]:
    gradcam_for_model(name, cfg, df_test, results, device);

## 16. Summary

In [ ]:
best_name = viz.best_model_name(results)
final_summary = {
    "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(cfg).items()},
    "individual_models": {n: r["metrics"] for n, r in results.items()},
    "ensemble": ensemble_metrics,
    "best_model": cfg.model_display[best_name],
}
with open(cfg.output_dir / "final_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2, default=str)
print(f"Best model: {cfg.model_display[best_name]}")
print(f"Outputs saved to: {cfg.output_dir.resolve()}")

# On Colab, download all results as a zip:
if "google.colab" in sys.modules:
    !zip -qr results.zip {cfg.output_dir}
    from google.colab import files
    files.download("results.zip")